In [2]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from pathlib import Path

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cuda


In [3]:
import torch
import sys

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA version:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

Python: 3.13.6 (tags/v3.13.6:4e66535, Aug  6 2025, 14:36:00) [MSC v.1944 64 bit (AMD64)]
PyTorch: 2.15.0+cu132
CUDA version: 13.2
CUDA available: True


In [4]:
# Find the dataset folder.
# This works if the notebook is run from either the project root or its code folder.
base_dir = Path.cwd()

if not (base_dir / "imagenette2-160").exists():
    if (base_dir.parent / "imagenette2-160").exists():
        base_dir = base_dir.parent

data_dir = base_dir / "imagenette2-160"

if not data_dir.exists():
    raise FileNotFoundError(
        f"Could not find {data_dir}. Update data_dir to point to your imagenette2-160 folder."
    )

# Transformations applied to each image
transform = transforms.Compose([
    transforms.Resize((160, 160)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Load training and validation datasets
train_dataset = datasets.ImageFolder(data_dir / "train", transform=transform)
val_dataset = datasets.ImageFolder(data_dir / "val", transform=transform)

print("Classes:", train_dataset.classes)
print("Training images:", len(train_dataset))
print("Validation images:", len(val_dataset))

Classes: ['n01440764', 'n02102040', 'n02979186', 'n03000684', 'n03028079', 'n03394916', 'n03417042', 'n03425413', 'n03445777', 'n03888257']
Training images: 9469
Validation images: 3925


In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

# Inspect one batch
images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)

Image batch shape: torch.Size([32, 3, 160, 160])
Label batch shape: torch.Size([32])


In [6]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv_stack = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.AdaptiveAvgPool2d((1, 1))
        )

        self.flatten = nn.Flatten()

        self.linear_stack = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 10)
        )

    def forward(self, x):
        x = self.conv_stack(x)
        x = self.flatten(x)
        logits = self.linear_stack(x)
        return logits

In [7]:
model = NeuralNetwork().to(device)

# Move one batch to the same device as the model
images, labels = next(iter(train_loader))
images = images.to(device)

# Run the images through the network
logits = model(images)

print("Output shape:", logits.shape)

Output shape: torch.Size([32, 10])


In [8]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [9]:
epochs = 20

for epoch in range(epochs):
    model.train()
    total_loss = 0
    correct = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = loss_fn(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()

    train_loss = total_loss / len(train_dataset)
    train_acc = correct / len(train_dataset)

    print(f"Epoch {epoch+1}/{epochs} | "
          f"Loss: {train_loss:.4f} | "
          f"Accuracy: {train_acc:.2%}")

Epoch 1/20 | Loss: 2.0524 | Accuracy: 25.81%
Epoch 2/20 | Loss: 1.8226 | Accuracy: 35.60%
Epoch 3/20 | Loss: 1.6450 | Accuracy: 42.94%
Epoch 4/20 | Loss: 1.5309 | Accuracy: 48.13%
Epoch 5/20 | Loss: 1.4265 | Accuracy: 51.67%
Epoch 6/20 | Loss: 1.3583 | Accuracy: 54.62%
Epoch 7/20 | Loss: 1.2805 | Accuracy: 57.73%
Epoch 8/20 | Loss: 1.2490 | Accuracy: 58.36%
Epoch 9/20 | Loss: 1.1893 | Accuracy: 60.47%
Epoch 10/20 | Loss: 1.1437 | Accuracy: 62.28%
Epoch 11/20 | Loss: 1.0947 | Accuracy: 63.96%
Epoch 12/20 | Loss: 1.0643 | Accuracy: 65.04%
Epoch 13/20 | Loss: 1.0224 | Accuracy: 66.52%
Epoch 14/20 | Loss: 0.9796 | Accuracy: 68.06%
Epoch 15/20 | Loss: 0.9517 | Accuracy: 68.95%
Epoch 16/20 | Loss: 0.9143 | Accuracy: 69.72%
Epoch 17/20 | Loss: 0.8885 | Accuracy: 70.99%
Epoch 18/20 | Loss: 0.8605 | Accuracy: 71.38%
Epoch 19/20 | Loss: 0.8390 | Accuracy: 72.31%
Epoch 20/20 | Loss: 0.8040 | Accuracy: 73.36%
